# RAGAS 평가 — ver2
### 주요 변경사항 (ver1 → ver2)
- `OllamaLLM` → **`ChatOllama`** 교체 (RAGAS 파싱 오류 수정)
- `safe_round()` 방어 코드 추가 (`type list doesn't define __round__` 오류 방지)
- 샘플별 NaN 비율 리포트 추가

In [1]:
# [셀1] 패키지 확인
# pip install ragas datasets langchain-ollama

In [2]:
# [셀2] 기존 all_results 불러오기
# ver1에서 저장한 experiment_results.json 사용
import json

DATA_DIR = r'D:\충북대\지능화캡스톤\data'  # ← 경로 확인

with open(f'{DATA_DIR}/experiment_results.json', 'r', encoding='utf-8') as f:
    all_results = json.load(f)

print(f'불러온 레코드 수: {len(all_results)}')

불러온 레코드 수: 498


In [3]:
# [셀3] RAGAS 평가 — ver2 핵심 수정: OllamaLLM → ChatOllama
from ragas import evaluate
from ragas import RunConfig
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# ✅ ver2 핵심 수정: OllamaLLM → ChatOllama
# ChatOllama는 chat 형식 메시지를 지원 → RAGAS JSON 파싱 성공률 대폭 향상
from langchain_ollama import ChatOllama, OllamaEmbeddings
from datasets import Dataset
import pandas as pd

# ── 판정용 LLM 설정 ──────────────────────────────────────────
JUDGE_MODEL = 'exaone3.5:7.8b'   # 판정 LLM (평가 대상 모델과 별개)
EMBED_MODEL  = 'exaone3.5:7.8b'  # 임베딩 모델

ragas_llm = LangchainLLMWrapper(
    ChatOllama(model=JUDGE_MODEL, temperature=0, num_ctx=4096)  # ✅ ChatOllama
)
ragas_emb = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(model=EMBED_MODEL)
)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
for m in metrics:
    m.llm = ragas_llm
    if hasattr(m, 'embeddings'):
        m.embeddings = ragas_emb

# ── 방어 함수: round 오류 방지 ────────────────────────────────
def safe_round(val, n=4):
    """NaN, list, None 등 비정상 값도 안전하게 처리"""
    try:
        return round(float(val), n)
    except Exception:
        return None

# ── 평가 루프 ─────────────────────────────────────────────────
ragas_scores = {}   # {model_name: score_dict}
ragas_dfs    = {}   # {model_name: result_dataframe}  ← NaN 분석용

for model_name in ['exaone', 'qwen3', 'llama31']:
    rag_records = [
        r for r in all_results
        if r['model'] == model_name
        and r['condition'] == 'rag'
        and r['context'].strip()
    ]
    print(f'\n[{model_name}] RAGAS 평가 중... ({len(rag_records)}개)')

    data = {
        'question':     [r['question']     for r in rag_records],
        'answer':       [r['pred_answer']  for r in rag_records],
        'contexts':     [[r['context']]    for r in rag_records],
        'ground_truth': [r['gold_answer']  for r in rag_records],
    }
    dataset = Dataset.from_dict(data)

    try:
        result = evaluate(
            dataset,
            metrics=metrics,
            run_config=RunConfig(
                timeout=180,
                max_workers=1,
                max_retries=2,
            )
        )

        # 결과 DataFrame 저장 (NaN 분석용)
        df = result.to_pandas()
        ragas_dfs[model_name] = df

        # NaN 비율 리포트
        nan_rates = df[['faithfulness','answer_relevancy',
                        'context_precision','context_recall']].isna().mean()
        print(f'  NaN 비율:')
        for col, rate in nan_rates.items():
            flag = ' ⚠️' if rate > 0.1 else ''
            print(f'    {col}: {rate:.1%}{flag}')

        # 점수 집계 (NaN 제외 평균)
        ragas_scores[model_name] = {
            'faithfulness':      safe_round(df['faithfulness'].mean()),
            'answer_relevancy':  safe_round(df['answer_relevancy'].mean()),
            'context_precision': safe_round(df['context_precision'].mean()),
            'context_recall':    safe_round(df['context_recall'].mean()),
        }
        print(f'  → {ragas_scores[model_name]}')

    except Exception as e:
        print(f'  [ERROR] {e}')
        ragas_scores[model_name] = {}

print('\nRagas 평가 완료')

d:\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hyebi\AppData\Local\Temp\ipykernel_56060\1265050190.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\hyebi\AppData\Local\Temp\ipykernel_56060\1265050190.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\hyebi\AppData\Local\Temp\ipykernel_56060\1265050190.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics


[exaone] RAGAS 평가 중... (83개)


Evaluating:  50%|█████     | 167/332 [18:20<16:33,  6.02s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt context_recall_classification_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[167]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating: 100%|██████████| 332/332 [36:40<00:00,  6.63s/it]


  NaN 비율:
    faithfulness: 0.0%
    answer_relevancy: 0.0%
    context_precision: 0.0%
    context_recall: 1.2%
  → {'faithfulness': 0.763, 'answer_relevancy': 0.2907, 'context_precision': 0.9036, 'context_recall': 0.9126}

[qwen3] RAGAS 평가 중... (83개)


Evaluating:  50%|█████     | 167/332 [16:50<13:42,  4.99s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt context_recall_classification_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[167]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating: 100%|██████████| 332/332 [34:20<00:00,  6.21s/it]


  NaN 비율:
    faithfulness: 0.0%
    answer_relevancy: 0.0%
    context_precision: 0.0%
    context_recall: 1.2%
  → {'faithfulness': 0.7966, 'answer_relevancy': 0.2858, 'context_precision': 0.9036, 'context_recall': 0.9126}

[llama31] RAGAS 평가 중... (83개)


Evaluating:  50%|█████     | 167/332 [15:22<13:35,  4.94s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt context_recall_classification_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[167]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating:  98%|█████████▊| 324/332 [31:21<01:03,  7.94s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output

  NaN 비율:
    faithfulness: 1.2%
    answer_relevancy: 0.0%
    context_precision: 0.0%
    context_recall: 1.2%
  → {'faithfulness': 0.7983, 'answer_relevancy': 0.2528, 'context_precision': 0.9036, 'context_recall': 0.9167}

Ragas 평가 완료


In [4]:
# [셀4] 결과 정리 및 저장
import pandas as pd
import json

# ROUGE-L / kw_recall은 ver1 eval_summary.csv에서 가져오기
ver1_summary = pd.read_csv(f'{DATA_DIR}/eval_summary.csv')

rows = []
for model_name in ['exaone', 'qwen3', 'llama31']:
    for condition in ['no_rag', 'rag']:
        row = ver1_summary[
            (ver1_summary['model'] == model_name) &
            (ver1_summary['condition'] == condition)
        ].iloc[0].to_dict()

        # RAG 조건이면 RAGAS 지표 업데이트
        if condition == 'rag' and model_name in ragas_scores and ragas_scores[model_name]:
            row.update(ragas_scores[model_name])

        rows.append(row)

summary_v2 = pd.DataFrame(rows)
print(summary_v2.to_string(index=False))

# 저장
out_csv  = f'{DATA_DIR}/eval_summary_v2.csv'
out_json = f'{DATA_DIR}/eval_results_v2.json'

summary_v2.to_csv(out_csv, index=False, encoding='utf-8-sig')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(ragas_scores, f, ensure_ascii=False, indent=2)

print(f'\nCSV 저장: {out_csv}')
print(f'JSON 저장: {out_json}')

  model condition  rouge_l  kw_recall  faithfulness  answer_relevancy  context_precision  context_recall
 exaone    no_rag   0.1141     0.3136           NaN               NaN                NaN             NaN
 exaone       rag   0.1891     0.5606        0.7630            0.2907             0.9036          0.9126
  qwen3    no_rag   0.0806     0.2597           NaN               NaN                NaN             NaN
  qwen3       rag   0.1698     0.4684        0.7966            0.2858             0.9036          0.9126
llama31    no_rag   0.0654     0.2461           NaN               NaN                NaN             NaN
llama31       rag   0.1423     0.4454        0.7983            0.2528             0.9036          0.9167

CSV 저장: D:\충북대\지능화캡스톤\data/eval_summary_v2.csv
JSON 저장: D:\충북대\지능화캡스톤\data/eval_results_v2.json
